In [24]:
import pandas as pd
import numpy as np
import os
import zipfile
import io


def extract_data():
    RAW_DATA_DIR = "./data/raw/walmart-recruiting-store-sales-forecasting"

    print("Loading data...")

    stores_path = os.path.join(RAW_DATA_DIR, "stores.csv")
    if os.path.exists(stores_path):
        stores = pd.read_csv(stores_path)
    else:
        raise FileNotFoundError("stores.csv not found in raw directory.")

    train = pd.read_csv(os.path.join(RAW_DATA_DIR, "train.csv"))
    features = pd.read_csv(os.path.join(RAW_DATA_DIR, "features.csv"))
    test = pd.read_csv(os.path.join(RAW_DATA_DIR, "test.csv"))

    print(f"Train shape: {train.shape}")
    print(f"Features shape: {features.shape}")
    print(f"Stores shape: {stores.shape}")
    print(f"Test shape: {test.shape}")

    print("Merging datasets...")

    # Merge train with features on Store and Date
    df_train_full = train.merge(features, on=['Store', 'Date'], how='left')

    # Merge with stores info on Store ID
    df_train_full = df_train_full.merge(stores, on='Store', how='left')

    # Do the same for test set
    df_test_full = test.merge(features, on=['Store', 'Date'], how='left')
    df_test_full = df_test_full.merge(stores, on='Store', how='left')

    # Convert Date column to datetime
    df_train_full['Date'] = pd.to_datetime(df_train_full['Date'])
    df_test_full['Date'] = pd.to_datetime(df_test_full['Date'])
    return df_train_full, df_test_full
df_train_full, df_test_full = extract_data()

Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...


In [26]:

# ---------------------------------------------------------
# 5. FEATURE ENGINEERING FUNCTION
# DON'T FORGET TO DROP DATE AND WEEKLY_SALES FROM FEATURES BEFORE MODELING!
# ---------------------------------------------------------
def create_features1(df, drop_columns=None):
    """
    Adds lag features, rolling stats, calendar features, encodes categoricals,
    and retains external macro-economic features.
    
    Args:
    df (pd.DataFrame): Input dataframe with 'Date', 'Store', 'Dept', 'Weekly_Sales', etc.
    drop_columns (list): List of column names to drop from the final output.
                        Defaults to ['Store', 'Dept', 'Id', 'Date', 'Type', 'IsHoliday', 'Weekly_Sales']
    
    Returns:
    pd.DataFrame: Processed dataframe ready for modeling.
    """
    df_out = df.copy()
    
    # Sort by Store, Dept, Date to ensure correct grouping for time-series features
    df_out = df_out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    
    group_cols = ['Store', 'Dept']
    
    # --- Lag Features ---
    df_out['sales_lag_1'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(1)
    df_out['sales_lag_2'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(2)
    df_out['sales_lag_4'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(4)
    df_out['sales_lag_52'] = df_out.groupby(group_cols)['Weekly_Sales'].shift(52) 
    
    # --- Rolling Statistics ---
    df_out['roll_mean_4'] = df_out.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
    )
    df_out['roll_std_4'] = df_out.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=4, min_periods=1).std()
    )
    df_out['roll_mean_8'] = df_out.groupby(group_cols)['Weekly_Sales'].transform(
        lambda x: x.shift(1).rolling(window=8, min_periods=1).mean()
    )
    
    # --- Calendar Features ---
    df_out['Year'] = df_out['Date'].dt.year
    df_out['Month'] = df_out['Date'].dt.month
    df_out['DayOfWeek'] = df_out['Date'].dt.dayofweek
    df_out['WeekOfYear'] = df_out['Date'].dt.isocalendar().week.astype(int)
    
    # --- Handle Markdowns ---
    markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
    existing_markdowns = [c for c in markdown_cols if c in df_out.columns]
    if existing_markdowns:
        df_out[existing_markdowns] = df_out[existing_markdowns].fillna(0)
    
    # --- Encode Categoricals ---
    type_map = {'A': 1, 'B': 2, 'C': 3}
    if 'Type' in df_out.columns:
        df_out['Type_encoded'] = df_out['Type'].map(type_map).fillna(0).astype(int)
    
    if 'IsHoliday' in df_out.columns:
        df_out['IsHoliday_int'] = df_out['IsHoliday'].astype(int)
    else:
        df_out['IsHoliday_int'] = 0
        
    # --- RETAIN EXTERNAL FEATURES ---
    # These columns already exist in df_out from the merge step.
    # We just need to make sure we DON'T drop them later.
    # Ensure they don't have NaNs (optional but good practice)
        external_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size']
    for col in external_cols:
        if col in df_out.columns:
            # Forward fill is better for macro data than filling with 0
            # bfill() handles any leading NaNs at the very start of the dataset
            df_out[col] = df_out[col].ffill().bfill() 

    # Markdowns should still be filled with 0 (no discount)
    markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
    existing_markdowns = [c for c in markdown_cols if c in df_out.columns]
    if existing_markdowns:
        df_out[existing_markdowns] = df_out[existing_markdowns].fillna(0)

    # --- Drop Unnecessary Columns ---
    if drop_columns is None:
        drop_columns = ['Store', 'Dept', 'Id', 'Type', 'IsHoliday']
    
    # Filter out columns that don't exist to avoid errors
    cols_to_drop = [col for col in drop_columns if col in df_out.columns]
    
    if cols_to_drop:
        df_out = df_out.drop(columns=cols_to_drop)
        
    return df_out
# 4. CHRONOLOGICAL SPLIT (TRAIN / VALIDATION)
# ---------------------------------------------------------
# Split date: Sep 1, 2012
VAL_START_DATE = '2012-09-01'
def train_and_val_split(df_with_features, val_start_date):
    
    df_train= df_with_features[df_with_features['Date'] < val_start_date].copy()
    df_val = df_with_features[df_with_features['Date'] >= val_start_date].copy()
    drop_cols = ['Weekly_Sales', 'Date']
    # Only drop if they exist
    drop_cols = [c for c in drop_cols if c in df_train.columns]

    X_train = df_train.drop(columns=drop_cols)
    y_train = df_train['Weekly_Sales']

    X_val = df_val.drop(columns=drop_cols)
    y_val = df_val['Weekly_Sales']

    # Handle any remaining object types (one-hot encode)
    object_cols = X_train.select_dtypes(include=['object']).columns
    if len(object_cols) > 0:
        print(f"One-hot encoding: {list(object_cols)}")
        X_train = pd.get_dummies(X_train, columns=object_cols)
        X_val = pd.get_dummies(X_val, columns=object_cols)

    # Align columns
    train_cols = X_train.columns
    val_cols = X_val.columns
    missing_in_val = set(train_cols) - set(val_cols)
    extra_in_val = set(val_cols) - set(train_cols)

    for col in missing_in_val:
        X_val[col] = 0
    for col in extra_in_val:
        X_val = X_val.drop(columns=[col])

    X_val = X_val.reindex(columns=train_cols)

    print("\nData Preparation Complete!")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_val shape: {X_val.shape}")

    # Now you can use X_train, y_train, X_val, y_val for modeling!
    print(X_train.head(), y_train.head())
    return X_train, y_train, X_val, y_val

# ---------------------------------------------------------
# example of use
df_with_features = create_features1(df_train_full)
X_train, y_train, X_val, y_val = train_and_val_split(df_with_features, VAL_START_DATE)



Data Preparation Complete!
X_train shape: (397841, 25)
X_val shape: (23729, 25)
   IsHoliday_x  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0        False        42.31       2.572        0.0        0.0        0.0   
1         True        38.51       2.548        0.0        0.0        0.0   
2        False        39.93       2.514        0.0        0.0        0.0   
3        False        46.63       2.561        0.0        0.0        0.0   
4        False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  ...           NaN   
4        0.0        0.0  211.350143         8.106  ...           NaN   

    roll_mean_4    roll_std_4